# Лабораторная работа № 1
## Валидация данных, обучение и интерпретация модели классификации изображений MNIST

**Вариант данных:** V2

**Язык:** Python 3.12  

**Фреймворки:** PyTorch и JAX



In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

from lab1_core import (
    ART_DIR, FIG_DIR, DATA_TRAIN, DATA_TEST, SEED, HIDDEN_DIM, BATCH_SIZE,
    EPOCHS, LEARNING_RATE, DROPOUT_GRID, ACTIVATIONS, N_PIXELS, N_CLASSES,
    ANOMALIES_CSV, VARIANT,
    load_mnist_csv, describe_dataset_structure, class_distribution,
    imbalance_metrics, image_statistics,
)

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
print('variant', VARIANT, 'batch', BATCH_SIZE)
print('hidden', HIDDEN_DIM, 'epochs', EPOCHS, 'lr', LEARNING_RATE)
print('dropout grid', DROPOUT_GRID)
print('activations', ACTIVATIONS)


variant V2 batch 512
hidden 256 epochs 12 lr 0.001
dropout grid [0.0, 0.2, 0.5]
activations ['relu', 'tanh', 'sigmoid']


## 2. Анализ датасета 

### 2.1. Структура исходного набора


In [2]:
x_all, y_all = load_mnist_csv(DATA_TRAIN)
x_test, y_test = load_mnist_csv(DATA_TEST)
print(json.dumps(describe_dataset_structure(x_all, y_all, DATA_TRAIN.name), ensure_ascii=False, indent=2))
print(json.dumps(describe_dataset_structure(x_test, y_test, 'mnist_test.csv'), ensure_ascii=False, indent=2))
print('уникальные метки train', np.unique(y_all))


{
  "name": "d2.csv",
  "n_objects": 60000,
  "n_features": 784,
  "image_shape": [
    28,
    28
  ],
  "n_classes": 10,
  "label_min": 0,
  "label_max": 9,
  "pixel_min": 0.0,
  "pixel_max": 255.0,
  "pixel_mean": 33.31842041015625,
  "dtype": "float32"
}
{
  "name": "mnist_test.csv",
  "n_objects": 10000,
  "n_features": 784,
  "image_shape": [
    28,
    28
  ],
  "n_classes": 10,
  "label_min": 0,
  "label_max": 9,
  "pixel_min": 0.0,
  "pixel_max": 255.0,
  "pixel_mean": 33.79122543334961,
  "dtype": "float32"
}
уникальные метки train [0 1 2 3 4 5 6 7 8 9]


### 2.2. Представление изображений и его ограничения

Изображение $28\times 28$ хранится как вектор $x \in \mathbb{R}^{784}$, полученный **построчной развёрткой** (row-major):

$$
x = (p_{1,1}, p_{1,2}, \dots, p_{1,28}, p_{2,1}, \dots, p_{28,28})
$$

**Что сохраняется:** значения интенсивности всех пикселей, глобальная яркость, «силуэт» цифры в фиксированной системе координат сетки.

**Что теряется / какие недостатки:**

- пространственная топология явно не кодируется: соседство пикселей на плоскости не отражено в соседстве координат вектора (пиксель `(1,28)` и `(2,1)` смежны в векторе, но не на изображении);
- нет инвариантности к сдвигу, масштабу, толщине штриха и небольшим поворотам;
- полносвязный слой вынужден заново «учить» локальность, в отличие от свёрток;
- фон и цифра смешаны в одном векторе без разделения объекта и контекста.

Именно поэтому двухслойный перцептрон на векторе 784 — базовая, но ограниченная модельб, она игнорирует 2D-структуру.


![Случайные примеры исходных изображений после восстановления формы 28×28](outputs_v2/figures/01_examples.png)

*Случайные примеры исходных изображений после восстановления формы 28×28*


![По одному примеру на каждый класс 0–9](outputs_v2/figures/02_one_per_class.png)

*По одному примеру на каждый класс 0–9*


### 2.3. Первичная проверка качества и понижение размерности


In [3]:
s = json.loads((ART_DIR / 'part1_summary.json').read_text(encoding='utf-8'))
print('пиксельный диапазон train:', s['structure_train']['pixel_min'], '…', s['structure_train']['pixel_max'])
print('пропуски/NaN: не обнаружены (min/max конечны, dtype float32)')
print('дубликаты:', s['duplicates'])
print('аномалий:', s['n_anomalies_full'], 'доля', round(s['anomaly_share_full'], 4))
print('причины:', s['anomaly_reason_counts'])


пиксельный диапазон train: 0.0 … 255.0
пропуски/NaN: не обнаружены (min/max конечны, dtype float32)
дубликаты: {'n_duplicate_groups': 0, 'n_extra_copies': 0, 'n_same_label_extra': 0, 'n_conflict_groups': 0, 'n_conflict_objects': 0, 'conflict_indices': [], 'same_label_indices': []}
аномалий: 572 доля 0.0095
причины: {'intensity_mean_outlier': 362, 'fill_outlier': 263, 'far_from_class_centroid': 234, 'intensity_std_outlier': 79}


Для визуализации структуры выборки на одной и той же подвыборке **4000** объектов построены PCA, t-SNE и UMAP.

- **PCA** — линейная проекция на направления максимальной дисперсии. Две компоненты объясняют лишь ≈17% дисперсии, классы сильно перекрываются. Метод быстрый и интерпретируемый, но для MNIST в 2D он слабо разделяет цифры.
- **t-SNE** — сохраняет локальные окрестности. Классы образуют компактные группы, видны типичные путаницы (4/9, 3/5, 7/9). Недостатки: вычислительная стоимость, зависимость от perplexity, плохая сохраняемость глобальной геометрии, нельзя применять «как модель» к новым точкам без пересчёта.
- **UMAP** — сохраняет и локальные кластеры, и часть глобальной структуры (например, «0» и «4» отделены, а «4–9–7» соседствуют). Работает быстрее t-SNE и устойчивее к параметрам.

**Вывод:** для качественного анализа разделимости классов на MNIST лучше **UMAP** (и t-SNE как контроль). PCA полезен как быстрый sanity-check, но не как основной инструмент.


![PCA, 2 компоненты](outputs_v2/figures/05_pca.png)

*PCA, 2 компоненты*


![t-SNE](outputs_v2/figures/06_tsne.png)

*t-SNE*


![UMAP](outputs_v2/figures/07_umap.png)

*UMAP*


![Распределения средней интенсивности и заполненности по классам](outputs_v2/figures/04_intensity_boxplots.png)

*Распределения средней интенсивности и заполненности по классам*


### 2.4. Аномальные объекты

Автоматический поиск использовал статистики изображения:

- средняя интенсивность, СКО, сумма, число ненулевых и доля активных пикселей (порог 10);
- евклидово расстояние до центроида своего класса;
- IQR-правило внутри класса;
- поиск полных дубликатов и конфликтов меток;
- проверка диапазона пикселей \([0, 255]\), NaN/Inf, пустых и почти заполненных изображений.

Найденные объекты сохранены в `{ANOMALIES_CSV.as_posix()}` вместе с исходной меткой, идентификатором строки и причиной.


In [4]:
anom = pd.read_csv(ANOMALIES_CSV)
display(anom.head(8))
print('всего аномалий', len(anom))
print(anom['reasons'].str.split(';').explode().value_counts())


,row_id,local_index,label,reasons,n_reasons,mean,std,active_frac,n_nonzero,centroid_dist
0,1618,1618,7,far_from_class_centroid;fill_outlier;intensity...,4,66.448982,106.221443,0.308673,244,2464.897461
1,4066,4066,1,far_from_class_centroid;fill_outlier;intensity...,4,49.882652,93.390755,0.260204,210,2523.827148
2,5972,5972,8,far_from_class_centroid;fill_outlier;intensity...,4,86.806122,116.259346,0.373724,294,2765.600830
3,6518,6518,9,far_from_class_centroid;fill_outlier;intensity...,4,68.271683,106.547600,0.321429,256,2514.205566
4,6885,6885,1,far_from_class_centroid;fill_outlier;intensity...,4,55.716835,100.940659,0.253827,201,2293.009277
5,8586,8586,9,far_from_class_centroid;fill_outlier;intensity...,4,66.844391,105.485016,0.322704,255,2785.745605
6,16376,16376,1,far_from_class_centroid;fill_outlier;intensity...,4,46.186226,98.205185,0.181122,142,2267.310547
7,25321,25321,8,far_from_class_centroid;fill_outlier;intensity...,4,88.711731,114.159500,0.414541,327,2914.569580


всего аномалий 572
reasons
intensity_mean_outlier     362
fill_outlier               263
far_from_class_centroid    234
intensity_std_outlier       79
Name: count, dtype: int64


![Топ аномалий по числу сработавших критериев](outputs_v2/figures/10_anomalies.png)

*Топ аномалий по числу сработавших критериев*


![Объекты далеко от центроида класса: нетипичный стиль написания](outputs_v2/figures/13_far_centroid.png)

*Объекты далеко от центроида класса: нетипичный стиль написания*


### 2.5. Распределение классов и дисбаланс


In [5]:
dist = pd.read_csv(ART_DIR / 'class_distribution_original.csv')
display(dist)
print('imbalance full:', json.dumps(s['imbalance_full'], ensure_ascii=False, indent=2))


,class,count,share
0,0,5923,0.098717
1,1,6742,0.112367
2,2,5958,0.099300
3,3,6131,0.102183
4,4,5842,0.097367
5,5,5421,0.090350
6,6,5918,0.098633
7,7,6265,0.104417
8,8,5851,0.097517
9,9,5949,0.099150


imbalance full: {
  "n": 60000,
  "min_class": 5,
  "max_class": 1,
  "min_count": 5421,
  "max_count": 6742,
  "imbalance_ratio": 1.243681977494927,
  "entropy": 2.3011591727032914,
  "normalized_entropy": 0.9993807306860915,
  "majority_share": 0.11236666666666667
}


![Гистограмма числа объектов по классам (исходный V2)](outputs_v2/figures/03_class_hist.png)

*Гистограмма числа объектов по классам (исходный V2)*


Дисбаланс **умеренный**: самый частый класс — `1` (6742), самый редкий — `5` (5421), imbalance ratio $$\approx 1.24$$. Нормированная энтропия $$\approx 0.999$$ идеальный баланс = 1. Для MNIST это типичная картина: единица пишется проще и встречается чаще. Существенного риска «забывания» редких классов нет, но выравнивание всё равно полезно как контролируемый фактор эксперимента.


### 2.6. Репрезентативность выборки

Сравнение train (`d2.csv`) и внешнего теста (`mnist_test.csv`) приведено в `representativeness.json` и в ячейке с `part1_summary.json`. Смотрите максимальное расхождение долей классов, среднюю яркость и сдвиг центроидов.


### 2.7. Сбалансированный и очищенный обучающие наборы

Стратифицированное разбиение `d2.csv` (seed=42):

| выборка | размер | роль |
|---|---|---|
| train | 51 000 | обучение |
| val | 9 000 | ранняя остановка и выбор гиперпараметров |
| test | 10 000 (`mnist_test.csv`) | **единая** финальная оценка всех моделей |

Обработка применяется **только к train**:

- **сбалансированный:** oversampling с возвращением до размера majority-класса → 57 310 объектов, ровно 5731 на класс;
- **очищенный:** удаление статистических аномалий, найденных на train → 50 520 объектов.

Val и test не балансируются и не чистятся, так как иначе сравнение обобщающей способности было бы смещённым.


![Сравнение распределений классов до и после балансировки/очистки](outputs_v2/figures/18_class_compare.png)

*Сравнение распределений классов до и после балансировки/очистки*


![Число объектов в исходном, сбалансированном и очищенном train](outputs_v2/figures/19_set_sizes.png)

*Число объектов в исходном, сбалансированном и очищенном train*


In [6]:
print('train', s['split'])
print('IR original', round(s['imbalance_train']['imbalance_ratio'], 3))
print('IR balanced', round(s['imbalance_balanced']['imbalance_ratio'], 3))
print('IR cleaned', round(s['imbalance_cleaned']['imbalance_ratio'], 3))


train {'train': 51000, 'val': 9000, 'test': 10000, 'balanced_train': 57310, 'cleaned_train': 50520, 'val_fraction': 0.15, 'stratified': True, 'seed': 42}
IR original 1.244
IR balanced 1.0
IR cleaned 1.237


## 3. Архитектура и обучение

### 3.1. Двухслойный перцептрон

Архитектура (одинаковая в PyTorch и JAX):

$$
h = \phi(W_1 x + b_1), \quad z = W_2 h + b_2,
$$

где $$ x \in \mathbb{R}^{784}, h \in \mathbb{R}^{256}, z \in \mathbb{R}^{10}$$ — **логиты**.

| слой              | вход | выход | параметры                    |
|-------------------|---|---|------------------------------|
| Linear 1          | 784 | 256 | $$784\cdot256 + 256 = 200\,960$$ |
| активация $$phi$$ | 256 | 256 | 0                            |
| Dropout $$p$$     | 256 | 256 | 0                            |
| Linear 2          | 256 | 10 | $$256\cdot10 + 10 = 2\,570$$ |
| **всего**         |  |  | **203 530**                  |

Инициализация: Xavier uniform, смещения нулевые.

### 3.2. Где применяется softmax

**Softmax не входит в `forward` модели.**

- **Обучение.** PyTorch: `nn.CrossEntropyLoss` = `LogSoftmax` + `NLLLoss`. JAX: `optax.softmax_cross_entropy_with_integer_labels`. Нормализация, эквивалентная softmax, выполняется **внутри функции потерь**.
- **Вывод.** Вероятности считаются явно: `softmax(logits)` (numpy) после получения логитов. Класс = `argmax`.


In [7]:
# Фрагмент архитектуры PyTorch (полная реализация — lab1_models.py)
import inspect, lab1_models
print(inspect.getsource(lab1_models.TwoLayerPerceptron))


class TwoLayerPerceptron(nn.Module):
    """Двухслойный перцептрон: Linear(784, H) -> Act -> Dropout -> Linear(H, 10).

    Выход — логиты. Softmax в модель не включён.
    """

    def __init__(self, hidden: int = HIDDEN_DIM, activation: str = "relu", dropout_p: float = 0.2):
        super().__init__()
        if activation not in ACTIVATIONS:
            raise ValueError(activation)
        self.activation_name = activation
        self.dropout_p = float(dropout_p)
        self.fc1 = nn.Linear(N_PIXELS, hidden)
        self.fc2 = nn.Linear(hidden, N_CLASSES)
        self.dropout = nn.Dropout(p=self.dropout_p)
        self._act = {"relu": F.relu, "tanh": torch.tanh, "sigmoid": torch.sigmoid}[activation]
        self._init_xavier()

    def _init_xavier(self) -> None:
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.zeros_(self.fc1.bias)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x: torch.Tensor) -> torc

### 3.3. Протокол эксперимента (единый для обоих фреймворков)

Сетка PyTorch: 3 набора × 3 активации × 3 dropout = 27 моделей.  
Сетка JAX: original+balanced полностью, cleaned — ReLU × dropout.


In [8]:
grid = pd.read_csv(ART_DIR / 'grid_all.csv')
print('экспериментов:', len(grid))
display(grid.sort_values(['framework','dataset','val_f1_macro'], ascending=[True, True, False]).groupby(['framework','dataset']).head(3))


экспериментов: 48


,framework,dataset,activation,dropout_p,tag,val_acc,val_f1_macro,val_f1_weighted,test_acc,test_f1_macro,test_f1_weighted,test_error_rate,mean_conf_correct,mean_conf_incorrect,n_params,train_time_sec,epochs_run,best_val_loss,device
37,jax,balanced,relu,0.2,jax_balanced_relu_p0.2,0.972889,0.972790,0.972897,0.9739,0.973725,0.973887,0.0261,0.983409,0.708079,203530,13.683931,12,0.096794,cpu
38,jax,balanced,relu,0.5,jax_balanced_relu_p0.5,0.970889,0.970789,0.970894,0.9729,0.972712,0.972876,0.0271,0.980391,0.702468,203530,10.428616,12,0.101437,cpu
36,jax,balanced,relu,0.0,jax_balanced_relu_p0.0,0.970333,0.970166,0.970327,0.9731,0.972933,0.973083,0.0269,0.982427,0.738554,203530,16.577704,12,0.103493,cpu
46,jax,cleaned,relu,0.2,jax_cleaned_relu_p0.2,0.972667,0.972623,0.972714,0.9761,0.976029,0.976108,0.0239,0.981202,0.697959,203530,11.663007,12,0.095803,cpu
47,jax,cleaned,relu,0.5,jax_cleaned_relu_p0.5,0.972222,0.972157,0.972236,0.9749,0.974808,0.974907,0.0251,0.977502,0.692995,203530,9.636878,12,0.102455,cpu
45,jax,cleaned,relu,0.0,jax_cleaned_relu_p0.0,0.972222,0.972143,0.972222,0.9751,0.975050,0.975094,0.0249,0.980618,0.704638,203530,10.136669,12,0.097478,cpu
28,jax,original,relu,0.2,jax_original_relu_p0.2,0.973556,0.973450,0.973545,0.9773,0.977200,0.977291,0.0227,0.980956,0.694895,203530,15.930110,12,0.089386,cpu
27,jax,original,relu,0.0,jax_original_relu_p0.0,0.971556,0.971494,0.971575,0.9762,0.976071,0.976207,0.0238,0.981131,0.704691,203530,15.595074,12,0.095239,cpu
29,jax,original,relu,0.5,jax_original_relu_p0.5,0.970778,0.970702,0.970774,0.9746,0.974434,0.974584,0.0254,0.977766,0.687804,203530,18.302871,12,0.098444,cpu
10,pytorch,balanced,relu,0.2,pytorch_balanced_relu_p0.2,0.974111,0.974001,0.974101,0.9750,0.974844,0.974989,0.0250,0.983508,0.712118,203530,5.349702,12,0.091136,cuda


### 3.4. Влияние dropout и функции активации


![PyTorch / original: val macro-F1 по активации и dropout](outputs_v2/figures/heat_pytorch_original.png)

*PyTorch / original: val macro-F1 по активации и dropout*


![PyTorch / balanced](outputs_v2/figures/heat_pytorch_balanced.png)

*PyTorch / balanced*


![PyTorch / cleaned](outputs_v2/figures/heat_pytorch_cleaned.png)

*PyTorch / cleaned*


![JAX / original: та же сетка](outputs_v2/figures/heat_jax_original.png)

*JAX / original: та же сетка*


![JAX / balanced](outputs_v2/figures/heat_jax_balanced.png)

*JAX / balanced*


![Ход обучения: разные активации при p=0.2 на original (PyTorch)](outputs_v2/figures/hist_activation_effect.png)

*Ход обучения: разные активации при p=0.2 на original (PyTorch)*


![Ход обучения: ReLU, p=0.0 vs p=0.5 на original (PyTorch)](outputs_v2/figures/hist_dropout_effect.png)

*Ход обучения: ReLU, p=0.0 vs p=0.5 на original (PyTorch)*


### 3.5. Лучшие модели трёх наборов


| фреймворк | набор | act | p | test acc | test macro-F1 | error rate | время, с | device |
|---|---|---|---|---|---|---|---|---|
| pytorch | original | relu | 0.2 | 0.9776 | 0.9775 | 0.0224 | 4.88 | cuda |
| pytorch | balanced | relu | 0.2 | 0.9750 | 0.9748 | 0.0250 | 5.35 | cuda |
| pytorch | cleaned | relu | 0.2 | 0.9753 | 0.9752 | 0.0247 | 4.58 | cuda |
| jax | original | relu | 0.2 | 0.9773 | 0.9772 | 0.0227 | 15.93 | cpu |
| jax | balanced | relu | 0.2 | 0.9739 | 0.9737 | 0.0261 | 13.68 | cpu |
| jax | cleaned | relu | 0.2 | 0.9761 | 0.9760 | 0.0239 | 11.66 | cpu |


In [9]:
best = pd.read_csv(ART_DIR / 'best_models.csv')
cols = ['framework','dataset','activation','dropout_p','val_acc','val_f1_macro','test_acc','test_f1_macro','test_f1_weighted','test_error_rate','mean_conf_correct','mean_conf_incorrect','train_time_sec','epochs_run','n_params','device']
display(best[cols])


,framework,dataset,activation,dropout_p,val_acc,val_f1_macro,test_acc,test_f1_macro,test_f1_weighted,test_error_rate,mean_conf_correct,mean_conf_incorrect,train_time_sec,epochs_run,n_params,device
0,pytorch,original,relu,0.2,0.975667,0.975590,0.9776,0.977484,0.977595,0.0224,0.980997,0.694022,4.884800,12,203530,cuda
1,pytorch,balanced,relu,0.2,0.974111,0.974001,0.9750,0.974844,0.974989,0.0250,0.983508,0.712118,5.349702,12,203530,cuda
2,pytorch,cleaned,relu,0.2,0.974667,0.974585,0.9753,0.975173,0.975290,0.0247,0.981107,0.705360,4.575344,12,203530,cuda
3,jax,original,relu,0.2,0.973556,0.973450,0.9773,0.977200,0.977291,0.0227,0.980956,0.694895,15.930110,12,203530,cpu
4,jax,balanced,relu,0.2,0.972889,0.972790,0.9739,0.973725,0.973887,0.0261,0.983409,0.708079,13.683931,12,203530,cpu
5,jax,cleaned,relu,0.2,0.972667,0.972623,0.9761,0.976029,0.976108,0.0239,0.981202,0.697959,11.663007,12,203530,cpu


![История обучения лучшей PyTorch-модели на original](outputs_v2/figures/hist_pytorch_original.png)

*История обучения лучшей PyTorch-модели на original*


![… на balanced](outputs_v2/figures/hist_pytorch_balanced.png)

*… на balanced*


![… на cleaned](outputs_v2/figures/hist_pytorch_cleaned.png)

*… на cleaned*


## 4. Количественная и качественная оценка

Для каждой из трёх моделей (original / balanced / cleaned) на едином тесте визуализируются:

- матрица ошибок 10×10;
- precision/recall/F1 по классам (в JSON-метриках);
- верные предсказания с высокой/низкой уверенностью;
- ошибки с высокой/низкой уверенностью;
- частые путаницы;
- объекты, где три модели не согласны;
- поведение на сохранённых аномалиях;
- окклюзионные карты ядер 1×1, 2×2, 3×3 с **общей** цветовой шкалой;
- карты весов первого слоя 28×28;
- доп. метод: градиентная чувствительность.


In [10]:
import glob
print('матрицы ошибок:')
for p in sorted(FIG_DIR.glob('cm_pytorch_*.png')):
    print(' ', p.name)


матрицы ошибок:
  cm_pytorch_balanced.png
  cm_pytorch_cleaned.png
  cm_pytorch_original.png


![Матрица ошибок PyTorch / original (test)](outputs_v2/figures/cm_pytorch_original.png)

*Матрица ошибок PyTorch / original (test)*


![Матрица ошибок PyTorch / balanced (test)](outputs_v2/figures/cm_pytorch_balanced.png)

*Матрица ошибок PyTorch / balanced (test)*


![Матрица ошибок PyTorch / cleaned (test)](outputs_v2/figures/cm_pytorch_cleaned.png)

*Матрица ошибок PyTorch / cleaned (test)*


### 4.1. Характерные предсказания (PyTorch)


![Original: верно, высокая уверенность](outputs_v2/figures/qual_pytorch_original_highconf_correct.png)

*Original: верно, высокая уверенность*


![Original: верно, низкая уверенность](outputs_v2/figures/qual_pytorch_original_lowconf_correct.png)

*Original: верно, низкая уверенность*


![Original: ошибка, высокая уверенность — самые опасные случаи](outputs_v2/figures/qual_pytorch_original_highconf_wrong.png)

*Original: ошибка, высокая уверенность — самые опасные случаи*


![Original: ошибка, низкая уверенность](outputs_v2/figures/qual_pytorch_original_lowconf_wrong.png)

*Original: ошибка, низкая уверенность*


![Balanced: ошибки с высокой уверенностью](outputs_v2/figures/qual_pytorch_balanced_highconf_wrong.png)

*Balanced: ошибки с высокой уверенностью*


![Cleaned: ошибки с высокой уверенностью](outputs_v2/figures/qual_pytorch_cleaned_highconf_wrong.png)

*Cleaned: ошибки с высокой уверенностью*


### 4.2. Расхождение трёх моделей и путаемые пары


![Объекты, на которых original / balanced / cleaned предсказывают по-разному](outputs_v2/figures/disagree_pytorch.png)

*Объекты, на которых original / balanced / cleaned предсказывают по-разному*


In [11]:
print(json.loads((ART_DIR/'confused_pytorch.json').read_text(encoding='utf-8')))
print(json.loads((ART_DIR/'confused_jax.json').read_text(encoding='utf-8')))


{'confused_pairs': [[7, 2, 13], [4, 9, 11], [7, 9, 9], [8, 3, 9], [2, 8, 8], [5, 3, 8]]}
{'confused_pairs': [[7, 2, 10], [7, 9, 10], [9, 4, 10], [2, 8, 9], [4, 9, 9], [8, 3, 8]]}


### 4.3. Аномалии, исключённые при очистке

На вход всех трёх моделей поданы объекты из `anomalies_{VARIANT.lower()}.csv`. Модель, обученная на original/balanced, видела часть этих примеров; cleaned — нет. Сравнение уверенности показывает, насколько фильтрация выбросов меняет чувствительность к нетипичному почерку.


In [12]:
ap = pd.read_csv(ART_DIR / 'anomalies_preds_pytorch.csv')
print('accuracy на аномалиях (PyTorch):')
for ds in ['original','balanced','cleaned']:
    print(f'  {ds}: {ap[ds+"_ok"].mean():.3f}, средняя уверенность {ap[ds+"_conf"].mean():.3f}')
display(ap[['row_id','label','reasons','original_pred','original_conf','balanced_pred','balanced_conf','cleaned_pred','cleaned_conf']].head(10))


accuracy на аномалиях (PyTorch):
  original: 0.921, средняя уверенность 0.921
  balanced: 0.911, средняя уверенность 0.936
  cleaned: 0.816, средняя уверенность 0.901


,row_id,label,reasons,original_pred,original_conf,balanced_pred,balanced_conf,cleaned_pred,cleaned_conf
0,1618,7,far_from_class_centroid;fill_outlier;intensity...,7,0.989616,7,0.998029,7,0.982296
1,4066,1,far_from_class_centroid;fill_outlier;intensity...,1,0.980013,1,0.979388,3,0.677556
2,5972,8,far_from_class_centroid;fill_outlier;intensity...,8,0.958094,8,0.984054,8,0.955695
3,6518,9,far_from_class_centroid;fill_outlier;intensity...,9,0.998864,9,0.999913,9,0.996825
4,6885,1,far_from_class_centroid;fill_outlier;intensity...,8,0.999162,8,0.999331,8,0.997257
5,8586,9,far_from_class_centroid;fill_outlier;intensity...,9,0.994331,9,0.999023,9,0.969517
6,16376,1,far_from_class_centroid;fill_outlier;intensity...,8,0.492716,1,0.537394,8,0.959446
7,25321,8,far_from_class_centroid;fill_outlier;intensity...,8,0.981179,8,0.839592,8,0.705749
8,26376,1,far_from_class_centroid;fill_outlier;intensity...,2,0.675729,2,0.711599,2,0.995425
9,26756,7,far_from_class_centroid;fill_outlier;intensity...,7,0.686383,7,0.875819,2,0.896795


![Предсказания трёх PyTorch-моделей на аномальных объектах](outputs_v2/figures/anom_preds_pytorch.png)

*Предсказания трёх PyTorch-моделей на аномальных объектах*


![То же для JAX](outputs_v2/figures/anom_preds_jax.png)

*То же для JAX*


### 4.4. Окклюзионные тепловые карты

Метод: окно \(k\times k\) последовательно заменяется нулём (нейтральный фон MNIST). Для каждой позиции считается падение вероятности **исходно предсказанного** класса. Значение окна равномерно наносится на все затронутые пиксели с усреднением перекрытий, поэтому карта всегда имеет размер 28×28. Для трёх моделей одного и того же изображения используется общая шкала.


![Окклюзия occ_pytorch_balanced_6400.png](outputs_v2/figures/occ_pytorch_balanced_6400.png)

*Окклюзия occ_pytorch_balanced_6400.png*


![Окклюзия occ_pytorch_balanced_8.png](outputs_v2/figures/occ_pytorch_balanced_8.png)

*Окклюзия occ_pytorch_balanced_8.png*


![Окклюзия occ_pytorch_balanced_899.png](outputs_v2/figures/occ_pytorch_balanced_899.png)

*Окклюзия occ_pytorch_balanced_899.png*


![Окклюзия occ_pytorch_cleaned_6400.png](outputs_v2/figures/occ_pytorch_cleaned_6400.png)

*Окклюзия occ_pytorch_cleaned_6400.png*


![Окклюзия occ_pytorch_cleaned_8.png](outputs_v2/figures/occ_pytorch_cleaned_8.png)

*Окклюзия occ_pytorch_cleaned_8.png*


![Окклюзия occ_pytorch_cleaned_899.png](outputs_v2/figures/occ_pytorch_cleaned_899.png)

*Окклюзия occ_pytorch_cleaned_899.png*


![Окклюзия occ_pytorch_original_6400.png](outputs_v2/figures/occ_pytorch_original_6400.png)

*Окклюзия occ_pytorch_original_6400.png*


![Окклюзия occ_pytorch_original_8.png](outputs_v2/figures/occ_pytorch_original_8.png)

*Окклюзия occ_pytorch_original_8.png*


![Окклюзия occ_pytorch_original_899.png](outputs_v2/figures/occ_pytorch_original_899.png)

*Окклюзия occ_pytorch_original_899.png*


![Частая путаница pair_pytorch_4_9.png](outputs_v2/figures/pair_pytorch_4_9.png)

*Частая путаница pair_pytorch_4_9.png*


![Частая путаница pair_pytorch_7_2.png](outputs_v2/figures/pair_pytorch_7_2.png)

*Частая путаница pair_pytorch_7_2.png*


![Частая путаница pair_pytorch_7_9.png](outputs_v2/figures/pair_pytorch_7_9.png)

*Частая путаница pair_pytorch_7_9.png*


### 4.5. Карты весов первого полносвязного слоя

Вектор весов нейрона длины 784 преобразуется в матрицу 28×28. Цветовая шкала `coolwarm`: положительные веса усиливают отклик при ярких пикселях (после нормализации \([0,1]\) и с учётом ReLU — только если линейная комбинация положительна). Отрицательные веса подавляют отклик в соответствующих областях. Показаны 16 нейронов с наибольшей \(\|w\|_2\).


![Веса первого слоя, PyTorch original](outputs_v2/figures/weights_pytorch_original.png)

*Веса первого слоя, PyTorch original*


![Веса первого слоя, PyTorch balanced](outputs_v2/figures/weights_pytorch_balanced.png)

*Веса первого слоя, PyTorch balanced*


![Веса первого слоя, PyTorch cleaned](outputs_v2/figures/weights_pytorch_cleaned.png)

*Веса первого слоя, PyTorch cleaned*


### 4.6. Дополнительный метод интерпретации: градиентная чувствительность

Для логита предсказанного (или истинного) класса считается

$$S_{ij} = \left| \frac{\partial z_c}{\partial x_{ij}} \right|.$$

В отличие от окклюзии, метод однопроходный и использует автоматическое дифференцирование фреймворка. Он показывает *локальную* чувствительность вокруг конкретного примера, тогда как окклюзия оценивает *конечное* влияние выключения патча. Совпадение горячих областей двух методов повышает доверие к интерпретации.


![Градиентная чувствительность sal_pytorch_6400.png](outputs_v2/figures/sal_pytorch_6400.png)

*Градиентная чувствительность sal_pytorch_6400.png*


![Градиентная чувствительность sal_pytorch_8.png](outputs_v2/figures/sal_pytorch_8.png)

*Градиентная чувствительность sal_pytorch_8.png*


![Градиентная чувствительность sal_pytorch_899.png](outputs_v2/figures/sal_pytorch_899.png)

*Градиентная чувствительность sal_pytorch_899.png*


## 5. PyTorch vs JAX

Сравнение выполнено в сопоставимых условиях: те же train/val/test, та же архитектура 784-256-10, Xavier, Adam \(10^{-3}\), batch 256, early stopping, одинаковые метрики.


![Лучшие модели: test accuracy и время обучения](outputs_v2/figures/compare_framework_bars.png)

*Лучшие модели: test accuracy и время обучения*


![Время обучения vs test accuracy](outputs_v2/figures/compare_time_vs_acc.png)

*Время обучения vs test accuracy*


![Матрица ошибок JAX / original](outputs_v2/figures/cm_jax_original.png)

*Матрица ошибок JAX / original*


![Карты весов первого слоя, JAX original](outputs_v2/figures/weights_jax_original.png)

*Карты весов первого слоя, JAX original*


In [13]:
print('Лучшие модели по фреймворкам')
display(best.pivot_table(index='dataset', columns='framework', values=['test_acc','test_f1_macro','train_time_sec']))
print('\nсреднее test_acc по всей сетке')
print(grid.groupby('framework')[['test_acc','val_f1_macro','train_time_sec']].mean())


Лучшие модели по фреймворкам


test_acc         test_f1_macro           train_time_sec          
framework      jax pytorch           jax   pytorch            jax   pytorch
dataset                                                                    
balanced    0.9739  0.9750      0.973725  0.974844      13.683931  5.349702
cleaned     0.9761  0.9753      0.976029  0.975173      11.663007  4.575344
original    0.9773  0.9776      0.977200  0.977484      15.930110  4.884800


среднее test_acc по всей сетке
           test_acc  val_f1_macro  train_time_sec
framework                                        
jax        0.964990      0.961916       13.682597
pytorch    0.963689      0.960734        5.136669


### Сравнительный анализ реализации

| аспект | PyTorch | JAX |
|---|---|---|
| модель | `nn.Module`, слои `Linear`/`Dropout` | явные параметры `W1,b1,W2,b2` в pytree |
| цикл обучения | императивный, `loss.backward(); opt.step()` | `jax.jit` + `jax.value_and_grad` + Optax |
| данные | тензоры на GPU, `randperm` + `index_select` | срезы `jnp.ndarray` после `permutation` |
| autodiff | tape (reverse-mode, динамический граф) | трансформации (`grad`, `jit`) на чистых функциях |
| dropout | `nn.Dropout` (inverted dropout, активен в `train()`) | `jax.random.bernoulli` + rescale |
| оптимизатор | `torch.optim.Adam` | `optax.adam` |
| устройство в этой работе | CUDA (RTX 5050) | CPU (официальный Windows-backend jaxlib) |
| воспроизводимость | `manual_seed` + GPU-resident shuffle | `PRNGKey(42)` на каждый шаг |

Метрики могут слегка расходится, вероятные причины: (1) устройство и численная точность GEMM; (2) порядок мини-пакетов (`DataLoader` vs `jax.random.permutation`); (3) детали Adam (ε, decay); (4) реализация dropout и генераторы случайных чисел; (5) early stopping на слегка разных val loss.



## 6. Итоговая сводка и выводы

1. **Данные V2** — вектор 784, пиксели \([0,255]\). Представление сохраняет интенсивности и теряет 2D-топологию.
2. **Проблемы качества** — дубликаты, дисбаланс и статистические аномалии зафиксированы в `part1_summary.json`.
3. **Репрезентативность** оценивается сравнением train/test (доли классов, яркость, центроиды) и визуализациями PCA/t-SNE/UMAP.
4. **Балансировка** — oversampling до majority; **очистка** — удаление IQR-выбросов и нетипичных объектов.
5. **Модель** — двухслойный перцептрон 784→256→10, 203 530 параметров, softmax внутри CE-loss.
6. **Dropout и активация.** ReLU стабильнее sigmoid; небольшой dropout сглаживает переобучение, слишком большой — мешает за 12 эпох.
7. **Три набора.** Очистка убирает трудные примеры; test остаётся «грязным» и проверяет устойчивость.
8. **Ошибки** типичны для MNIST: близкие по начертанию пары (4/9, 3/5, 7/9), нетипичный почерк.
9. **Аномалии:** original/balanced увереннее на выбросах; cleaned осторожнее — эффект фильтрации.
10. **Окклюзия и веса** показывают опору на штрихи цифры, а не на фон; градиентные карты согласуются с окклюзией.
11. **PyTorch и JAX** дают близкое качество; PyTorch на этой машине быстрее за счёт GPU-resident батчей.
12. **Ограничения:** нет пространственной инвариантности; аномалии размечены эвристиками. Улучшения: свёртки, аугментации, калибровка вероятностей.


## 7. Ответы на вопросы 

**1. Как 28×28 превращается в вектор и что теряется?**  
Изображение 28×28 разворачивается построчно в вектор 784. Сохраняются интенсивности всех пикселей. Теряется явная двумерная соседство/топология и инвариантность к сдвигу и повороту.

**2. Выход первого слоя, линейность, роли весов и смещений.**  
Первый слой: h0 = W1 x + b1. Это аффинное (линейное плюс смещение) преобразование: строки W1 шаблоны 28×28, b1 порог. Без последующей нелинейности вся сеть осталась бы линейной.

**3. Почему два линейных слоя без активации — всё ещё линейная модель?**  
Композиция двух линейных отображений линейна: W2(W1 x + b1)+b2 = (W2 W1)x + const. Активация между слоями необходима, чтобы расширить класс реализуемых функций.

**4. Функция потерь, градиент, почему сеть начинает узнавать цифры.**  
Cross-entropy сравнивает softmax-вероятность истинного класса с единицей. Градиент этой величины по весам указывает направление, увеличивающее логит верного класса. Итерации Adam сдвигают шаблоны первого слоя к штрихам цифр.

**5. Вероятность softmax и запоминание vs обобщение.**  
$$\mathrm{softmax}(z)_c$$ — нормированный вес логита, а не истинная вероятность ошибки в мире. Модель может быть уверенно неправа. Обобщение видно по val/test, которые не участвовали в градиенте; запоминание — по большому зазору train accuracy ≫ test accuracy. Dropout, ранняя остановка и сравнение original/cleaned как раз отличают эти режимы.


## 8. Как воспроизвести

```text
python -m venv .venv
.venv\Scripts\python.exe -m pip install -r requirements.txt
.venv\Scripts\python.exe run_part1.py
.venv\Scripts\python.exe run_train.py
```

